In [14]:
from vnpy.trader.optimize import OptimizationSetting
from vnpy_spreadtrading.backtesting import BacktestingEngine
from vnpy_spreadtrading.strategies.statistical_arbitrage_strategy import (
    StatisticalArbitrageStrategy
)
from vnpy_spreadtrading.base import LegData, SpreadData
from datetime import datetime

In [15]:
near_symbol = "ag2604.SHFE"
far_symbol = "ag2606.SHFE"
spread = SpreadData(
    name="IF-Spread",
    legs=[LegData(near_symbol), LegData(far_symbol)],
    variable_symbols={"A": near_symbol, "B": far_symbol},
    variable_directions={"A": 1, "B": -1},
    price_formula="A-B",
    trading_multipliers={near_symbol: 1, far_symbol: 1},
    active_symbol=near_symbol,
    min_volume=1,
    compile_formula=False                          # 回测时不编译公式，compile_formula传False，从而支持多进程优化
)

In [16]:
engine = BacktestingEngine()
engine.set_parameters(
    spread=spread,
    interval="1m",
    start=datetime(2025, 12, 1),
    end=datetime(2025, 12, 5),
    rate=0,
    slippage=1,
    size=300,
    pricetick=0.2,
    capital=1_000_000,
)
engine.add_strategy(StatisticalArbitrageStrategy, {})

In [19]:
engine.load_data()
engine.run_backtesting()
df = engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

2026-01-23 20:50:27.117499	开始加载历史数据
2026-01-23 20:50:27.256808	历史数据加载完成，数据量：2625
2026-01-23 20:50:27.276854	策略初始化完成
2026-01-23 20:50:27.276854	开始回放历史数据
2026-01-23 20:50:27.301370	历史数据回放结束
2026-01-23 20:50:27.301370	开始计算逐日盯市盈亏
2026-01-23 20:50:27.303374	逐日盯市盈亏计算完成
2026-01-23 20:50:27.303374	开始计算策略统计指标
2026-01-23 20:50:27.305374	------------------------------
2026-01-23 20:50:27.305374	首个交易日：	2025-12-01
2026-01-23 20:50:27.305374	最后交易日：	2025-12-05
2026-01-23 20:50:27.305374	总交易日：	5
2026-01-23 20:50:27.305374	盈利交易日：	2
2026-01-23 20:50:27.305374	亏损交易日：	3
2026-01-23 20:50:27.305374	起始资金：	1,000,000.00
2026-01-23 20:50:27.305374	结束资金：	512,800.00
2026-01-23 20:50:27.305374	总收益率：	-48.72%
2026-01-23 20:50:27.305374	年化收益：	-2,338.56%
2026-01-23 20:50:27.305374	最大回撤: 	-532,200.00
2026-01-23 20:50:27.305374	百分比最大回撤: -60.19%
2026-01-23 20:50:27.305374	最长回撤天数: 	3
2026-01-23 20:50:27.305374	总盈亏：	-487,200.00
2026-01-23 20:50:27.305374	总手续费：	0.00
2026-01-23 20:50:27.305374	总滑点：	622,200.00
2026-01-23 20:5

In [20]:
for trade in engine.trades.values():
    print(trade)

TradeData(gateway_name='BACKTESTING', extra=None, symbol='IF-Spread', exchange=<Exchange.LOCAL: 'LOCAL'>, orderid='16', tradeid='1', direction=<Direction.LONG: 'Long'>, offset=<Offset.NONE: ''>, price=15.0, volume=10, datetime=datetime.datetime(2025, 12, 1, 9, 16, tzinfo=zoneinfo.ZoneInfo(key='Asia/Shanghai')))
TradeData(gateway_name='BACKTESTING', extra=None, symbol='IF-Spread', exchange=<Exchange.LOCAL: 'LOCAL'>, orderid='17', tradeid='2', direction=<Direction.SHORT: 'Short'>, offset=<Offset.NONE: ''>, price=17.0, volume=10, datetime=datetime.datetime(2025, 12, 1, 9, 23, tzinfo=zoneinfo.ZoneInfo(key='Asia/Shanghai')))
TradeData(gateway_name='BACKTESTING', extra=None, symbol='IF-Spread', exchange=<Exchange.LOCAL: 'LOCAL'>, orderid='18', tradeid='3', direction=<Direction.LONG: 'Long'>, offset=<Offset.NONE: ''>, price=13.0, volume=10, datetime=datetime.datetime(2025, 12, 1, 9, 43, tzinfo=zoneinfo.ZoneInfo(key='Asia/Shanghai')))
TradeData(gateway_name='BACKTESTING', extra=None, symbol='I

In [21]:
setting = OptimizationSetting()
setting.set_target("sharpe_ratio")
setting.add_parameter("boll_window", 10, 30, 1)
setting.add_parameter("boll_dev", 1, 3, 1)

engine.run_ga_optimization(setting)

2026-01-23 20:50:41.902897	Starting optimization with genetic algorithm
2026-01-23 20:50:41.902897	Parameter optimization space: 63
2026-01-23 20:50:41.902897	Total number of populations per generation: 100
2026-01-23 20:50:41.902897	Number of good filters: 80
2026-01-23 20:50:41.902897	Number of iterations: 30
2026-01-23 20:50:41.902897	Crossover probability: 95%
2026-01-23 20:50:41.902897	Mutation probability: 5%
2026-01-23 20:50:41.902897	个体突变概率：100%
gen	nevals
0  	100   
1  	100   
2  	100   
3  	100   
4  	100   
5  	100   
6  	100   
7  	100   
8  	100   
9  	100   
10 	100   
11 	100   
12 	100   
13 	100   
14 	100   
15 	100   
16 	100   
17 	100   
18 	100   
19 	100   
20 	100   
21 	100   
22 	100   
23 	100   
24 	100   
25 	100   
26 	100   
27 	100   
28 	100   
29 	100   
30 	100   
2026-01-23 20:50:48.407572	Optimization with genetic algorithm complete, 6 seconds elapsed
2026-01-23 20:50:48.500703	参数：{'boll_window': 26, 'boll_dev': 3}, 目标：7.252134106529443
2026-01-23 2

[({'boll_window': 26, 'boll_dev': 3},
  np.float64(7.252134106529443),
  {'start_date': datetime.date(2025, 12, 1),
   'end_date': datetime.date(2025, 12, 5),
   'total_days': 5,
   'profit_days': 2,
   'loss_days': 3,
   'capital': 1000000,
   'end_balance': np.float64(1043200.0),
   'max_drawdown': np.float64(-10800.0),
   'max_ddpercent': np.float64(-1.0246679316888045),
   'max_drawdown_duration': 2,
   'total_net_pnl': np.float64(43200.0),
   'daily_net_pnl': np.float64(8640.0),
   'total_commission': np.float64(0.0),
   'daily_commission': np.float64(0.0),
   'total_slippage': np.float64(19800.0),
   'daily_slippage': np.float64(3960.0),
   'total_turnover': np.float64(2669994000.0),
   'daily_turnover': np.float64(533998800.0),
   'total_trade_count': np.int64(32),
   'daily_trade_count': np.float64(6.4),
   'total_return': np.float64(4.3199999999999905),
   'annual_return': np.float64(207.35999999999956),
   'daily_return': np.float64(0.9541508978350068),
   'return_std': np.fl

In [22]:
engine.run_bf_optimization(setting)

2026-01-23 20:51:46.520108	Starting optimization with brute force algorithm
2026-01-23 20:51:46.520108	Parameter optimization space: 63


100%|██████████| 63/63 [00:03<00:00, 19.31it/s]


2026-01-23 20:51:49.946696	Optimization with brute force algorithm complete, 3 seconds elapsed
2026-01-23 20:51:50.215265	参数：{'boll_window': 26, 'boll_dev': 3}, 目标：7.252134106529443
2026-01-23 20:51:50.216267	参数：{'boll_window': 30, 'boll_dev': 2}, 目标：7.21274361240929
2026-01-23 20:51:50.216267	参数：{'boll_window': 27, 'boll_dev': 3}, 目标：7.125770311054317
2026-01-23 20:51:50.216267	参数：{'boll_window': 13, 'boll_dev': 3}, 目标：6.92820323027551
2026-01-23 20:51:50.216267	参数：{'boll_window': 28, 'boll_dev': 3}, 目标：6.133607860991462
2026-01-23 20:51:50.216267	参数：{'boll_window': 30, 'boll_dev': 3}, 目标：6.02704837845868
2026-01-23 20:51:50.216267	参数：{'boll_window': 25, 'boll_dev': 3}, 目标：5.720057951493069
2026-01-23 20:51:50.216267	参数：{'boll_window': 21, 'boll_dev': 3}, 目标：5.0265195985694975
2026-01-23 20:51:50.216267	参数：{'boll_window': 29, 'boll_dev': 3}, 目标：5.002913837478025
2026-01-23 20:51:50.216267	参数：{'boll_window': 24, 'boll_dev': 3}, 目标：4.997701060325047
2026-01-23 20:51:50.216267	参数：{'boll_

[({'boll_window': 26, 'boll_dev': 3},
  np.float64(7.252134106529443),
  {'start_date': datetime.date(2025, 12, 1),
   'end_date': datetime.date(2025, 12, 5),
   'total_days': 5,
   'profit_days': 2,
   'loss_days': 3,
   'capital': 1000000,
   'end_balance': np.float64(1043200.0),
   'max_drawdown': np.float64(-10800.0),
   'max_ddpercent': np.float64(-1.0246679316888045),
   'max_drawdown_duration': 2,
   'total_net_pnl': np.float64(43200.0),
   'daily_net_pnl': np.float64(8640.0),
   'total_commission': np.float64(0.0),
   'daily_commission': np.float64(0.0),
   'total_slippage': np.float64(19800.0),
   'daily_slippage': np.float64(3960.0),
   'total_turnover': np.float64(2669994000.0),
   'daily_turnover': np.float64(533998800.0),
   'total_trade_count': np.int64(32),
   'daily_trade_count': np.float64(6.4),
   'total_return': np.float64(4.3199999999999905),
   'annual_return': np.float64(207.35999999999956),
   'daily_return': np.float64(0.9541508978350068),
   'return_std': np.fl